### Importing packages and loading data

In [ ]:
import pandas as pd
import numpy as np
from gensim.models import KeyedVectors
from transformers import pipeline, AutoTokenizer, AutoModel
import torch
from scipy.stats import ttest_ind

# --- Load words ---
df = pd.read_csv("gender_words.csv")
words = df["word"].dropna().unique().tolist()

### WEAT

In [ ]:
# --- WEAT (Static Embeddings) ---
print("\n=== WEAT Using FastText ===")
model_static = KeyedVectors.load_word2vec_format("cc.da.300.vec")

male_words = ['mand', 'han', 'dreng', 'far']
female_words = ['kvinde', 'hun', 'pige', 'mor']

def avg_cos_sim(word, group):
    try:
        return np.mean([model_static.similarity(word, g) for g in group])
    except KeyError:
        return np.nan

def weat_score(word):
    male_sim = avg_cos_sim(word, male_words)
    female_sim = avg_cos_sim(word, female_words)
    return male_sim - female_sim

df["weat_bias_score"] = [weat_score(w) for w in words]

### BERT Masked Word Prediction

In [ ]:
# --- Masked Word Prediction (Contextual) ---
print("\n=== BERT Masked Word Prediction ===")
fill_mask = pipeline("fill-mask", model="Maltehb/danish-bert-botxo")

example_template_fem = "Hun er meget [MASK]."
example_template_masc = "Han er meget [MASK]."

print("Top predictions for neutral template:")
print("Female:", [res['token_str'] for res in fill_mask(example_template_fem)])
print("Male:", [res['token_str'] for res in fill_mask(example_template_masc)])

### Contextual Embedding Similarity

In [ ]:
# --- Contextual Embedding Similarity ---
print("\n=== Contextual Embedding Similarity ===")
tokenizer = AutoTokenizer.from_pretrained("Maltehb/danish-bert-botxo")
model = AutoModel.from_pretrained("Maltehb/danish-bert-botxo")

def get_word_embedding(sentence, target_word):
    tokens = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**tokens)
    token_ids = tokens['input_ids'][0]
    target_token = tokenizer.tokenize(target_word)
    try:
        idx = (token_ids == tokenizer.convert_tokens_to_ids(target_token[0])).nonzero(as_tuple=True)[0][0]
        return outputs.last_hidden_state[0, idx, :]
    except IndexError:
        return None

cosine = torch.nn.functional.cosine_similarity

def contextual_bias(word):
    s_fem = f"Hun er en meget {word} person."
    s_masc = f"Han er en meget {word} person."
    emb_fem = get_word_embedding(s_fem, word)
    emb_masc = get_word_embedding(s_masc, word)
    if emb_fem is not None and emb_masc is not None:
        return cosine(emb_fem, emb_masc, dim=0).item()
    else:
        return np.nan

df["contextual_similarity"] = [contextual_bias(w) for w in words]

### Save the output(s)

In [ ]:
# --- Save Output ---
df.to_csv("gender_bias_results.csv", index=False)
print("\n🎉 Analysis complete! Results saved to 'gender_bias_results.csv'")